In [ ]:
!pip -q install tensorflow

In [ ]:
# BETTER TO TRAIN WITH TPU v5e-1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from tqdm import tqdm
import joblib
import json
import glob
import pickle
import sys
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # ignore warnings

from google.colab import files
from zipfile import ZipFile

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Layer, BatchNormalization, Dropout, LayerNormalization, Activation
from tensorflow.keras.activations import relu, tanh, sigmoid, swish, elu
from tensorflow.keras.utils import plot_model
#from sklearn.model_selection import train_test_split  # not used
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import  StandardScaler

os.makedirs('resultados', exist_ok=True)  # to save all results

**MAIN GENERAL VARIABLES**

In [ ]:
#####################################################################################################################
epocas = 450                 # max training epochs (stops early - we just set extra)
batch = 8                    # batch size

early_stop_patience = 60     # epochs for the stopping criteria
early_stop_min_delta = 3e-4  # counts as no improvement if change is < min_delta, to stop training
start_earlystop = 10         # epochs to wait before checking the stop criteria

reduce_lr_factor = 0.7       # learning rate reduction factor
reduce_lr_patience = 10       # epochs to wait before checking for improvement
reduce_lr_min_delta = 3e-4   # counts as no improvement if change is < min_delta, to drop lr
lr_min = 5e-9                # min learning rate
initial_lr = 3e-4            # starting learning rate
cool_down = 2                # epochs to wait before counting again after dropping lr

Loss = 'mean_squared_error'  # loss variable
loss_Hb = Huber(delta=0.02)
MAE = 'mae'                  # extra metric for training

dpi = 300                    # for saving plots
IQR_factor = 1.5             # to spot outliers (preprocessing)
#####################################################################################################################

**UPLOAD FILES**

In [ ]:
# UPLOAD CNN.py MODULE

files.upload()

from CNN import crear_modelo, SelfAttention, set_attention_wavelengths
from CNN import CNN_L2, Conv_Act, Dense_Act, Filters, Kernels, Pool, Att_Units, Att_Heads, Dense_Units

# for another model later
#from Transformer1D import crear_modelo, SelfAttention, set_attention_wavelengths
#from Transformer1D import CNN_L2, Conv_Act, Dense_Act, Filters, Kernels, Pool, Att_Units, Att_Heads, Dense_Units

In [ ]:
# UPLOAD THE REQUIRED FILES

print("Upload the files: Espectros.zip, wavelengths.csv and temperature_values.csv")
uploaded = files.upload()


# unzip spectra
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with ZipFile(fname, 'r') as zip_ref:
            zip_ref.extractall()
        print(f"Unzipped: {fname}")

In [ ]:
# set paths, cropping range, and number of spectra for Train/Test and Train/Val split

ruta_espectros = './Espectros_rename'  # spectra CSVs are inside this folder named (1.csv, 2.csv, 3.csv, ...)
ruta_waves = './wavelengths.csv'
ruta_tabla = './temperature_values.csv'


# λ range
#------------------------------------------------------------------------------------------------#
λ_min = float(input("Enter the Minimum wavelength in [nm]: "))
λ_max = float(input("Enter the Maximum wavelength in [nm]: "))

print(f"Selected wavelength range: {λ_min} nm, {λ_max} nm")
print(f"Files ready to process in: {ruta_espectros}")


#------------------------------------------------------------------------------------------------#
#------------------------------------------------------------------------------------------------#
# number of spectra for train + test (in order)
num_train_test = int(input("Enter the spectrum number to split Train + Test. Starting from 1: "))

# number of spectra for train + val (in order)
num_train_val = int(input("Enter the spectrum number where Val starts within Train: "))

**SPECTRA CROPPING AND CLEANING**

In [ ]:
# read wavelengths
wave_df = pd.read_csv(ruta_waves, header=None)
wavelengths = wave_df.iloc[:,0].values

# read temperatures
tabla_df     = pd.read_csv(ruta_tabla, header=None)
temperaturas = np.round(tabla_df.iloc[:, 0].values, 3)

#------------------------------------------------------------------------------------------------#
# spectra cropping
espectros = []
csv_files = glob.glob(os.path.join(ruta_espectros, '*.csv'))

for i in tqdm(range(len(temperaturas)), desc="Processing spectra...", ncols=100, leave=False):

    inten = pd.read_csv(os.path.join(ruta_espectros, f'{i+1}.csv'), # read intensities
                        header=None).iloc[:,0].values

    # 2d array [λ, I]
    data = np.vstack((wavelengths, inten)).T

    mask = (data[:,0] >= λ_min) & (data[:,0] <= λ_max)  # crop λ_min - λ_max
    espectros.append(data[mask])

X = np.array(espectros)    # shape: (M, Ni, 2)
y = np.array(temperaturas) # shape: (M,)

#------------------------------------------------------------------------------------------------#
set_attention_wavelengths(wavelengths[(wavelengths>=λ_min) & (wavelengths<=λ_max)]) # for the attention module!!
#------------------------------------------------------------------------------------------------#


#------------------------------------------------------------------------------------------------#
print(f"Number of processed spectra:  {len(X)}")
print(f"Number of loaded temperatures:  {len(y)}")

In [ ]:
# VERIFY CROPPING

idx = np.random.randint(1, len(X) + 1)  # show a random spectrum

plt.figure(figsize=(8,4))
plt.plot(X[idx,:,0], X[idx,:,1], label=f'Spectrum {idx}')
plt.xlabel('λ [nm]')
plt.title(f'Cropped spectrum {idx} ({λ_min}–{λ_max} nm)')
plt.show()

In [ ]:
# REMOVE NULL DATA (IF ANY)

print("Original data count:", X.shape[0])
#------------------------------------------------------------------------------------------------#

mask_nan = ~np.isnan(X).any(axis=(1, 2))  # X is 3d
X, y = X[mask_nan], y[mask_nan]
print(f"After removing NaNs: {X.shape[0]} samples (dropped {np.sum(~mask_nan)})")

#------------------------------------------------------------------------------------------------#
# REMOVE OUTLIERS IN Y (IF ANY, USING IQR)

Q1, Q3 = np.percentile(y, [25, 75]) # calculate iqr to drop outliers
IQR = Q3 - Q1
lower_bound = Q1 - IQR_factor * IQR # check if we should keep this 1.5 factor or change it!!
upper_bound = Q3 + IQR_factor * IQR

mask_outliers = (y >= lower_bound) & (y <= upper_bound)
print(f"Outliers detected and dropped: {np.sum(~mask_outliers)}")

X, y = X[mask_outliers], y[mask_outliers]
print(f"After removing outliers: {X.shape[0]} samples")

#------------------------------------------------------------------------------------------------#
# CHECK FINAL DIMENSIONS
print("X dimension: ", X.shape)
print("y dimension: ", y.shape)

**DATA SPLIT AND SCALING**

In [ ]:
# split for training and scaling

#------------------------------------------------------------------------------------------------#
# MANUAL SPLIT (TRAIN AND TEST):
X_train, X_test = X[:num_train_test], X[num_train_test:]    # starts from 0 ? (check)
y_train, y_test = y[:num_train_test], y[num_train_test:]

# MANUAL SPLIT (TRAIN AND VAL):
X_train, X_val = X_train[:num_train_val], X_train[num_train_val:]
y_train, y_val = y_train[:num_train_val], y_train[num_train_val:]


# SCALING:
#------------------------------------------------------------------------------------------------#

# scale labels (T):
scaler_y = StandardScaler()  # centered at 0 with variance 1
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val   = scaler_y.transform( y_val.reshape(-1, 1))
y_test  = scaler_y.transform( y_test .reshape(-1, 1))

# scale intensities:
scaler_x = StandardScaler()
X_train = scaler_x.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_val   = scaler_x.transform(X_val.reshape(-1, 1)).reshape(X_val.shape)
X_test = scaler_x.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
# the wavelengths.csv file is always the same. if using a different one use (-1,2) in the X_train, X_test reshapes


scale_T = float(scaler_y.scale_[0])  # std of T (°C) - to track training in °C


# tensor shapes
#------------------------------------------------------------------------------------------------#
print("X_train shape: ", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_test shape: ", y_test.shape)
print("X_val shape: ", X_val.shape)
print("y_val shape: ", y_val.shape)

#------------------------------------------------------------------------------------------------#
print('Points per spectrum:', X_train.shape[1])

**MODEL COMPILATION**



In [ ]:
# COMPILE THE MODEL:

#------------------------------------------------------------------------------------------------#
input_shape = (X_train.shape[1], X_train.shape[2])  # for convolutional
model = crear_modelo(input_shape)
#------------------------------------------------------------------------------------------------#

#------------------------------------------------------------------------------------------------#
# metrics to track training in °C (still training on scaled data)
def rmse_C(y_true, y_pred):
    return tf.sqrt( (scale_T**2) * tf.reduce_mean(tf.square(y_true - y_pred)) )
def mae_C(y_true, y_pred):
    return scale_T * tf.reduce_mean(tf.abs(y_true - y_pred))
#------------------------------------------------------------------------------------------------#


# compile model
model.compile(optimizer=Adam(learning_rate= initial_lr, clipnorm=1.0), loss= loss_Hb, metrics=[MAE, rmse_C, mae_C],
                 jit_compile= False, run_eagerly= False)  # add metrics here
# metrics that can be added during compilation go here


model.summary()
#------------------------------------------------------------------------------------------------#

**TRAINING**

In [ ]:
# callbacks
#---------------------------------------------------------------------------------------------------------------#
# checkpoint
ckpt = ModelCheckpoint(filepath="resultados/best_val_mae_c.weights.h5", monitor="val_mae_c", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0 )

# earlystopping (stopping criteria)
early_stop = EarlyStopping(monitor='val_mae_c', patience= early_stop_patience, # val_loss can also be used
    min_delta= early_stop_min_delta, start_from_epoch= start_earlystop,
    restore_best_weights=True, verbose=1, mode= 'min' )

# drop learning rate during training
reduce_lr = ReduceLROnPlateau(monitor='val_mae_c', factor= reduce_lr_factor, # val_loss can also be used
    patience= reduce_lr_patience, min_delta= reduce_lr_min_delta, min_lr= lr_min,
    restore_best_weights=True, cooldown= cool_down, verbose=1, mode= 'min' )
#---------------------------------------------------------------------------------------------------------------#


# START TRAINING
#------------------------------------------------------------------------------------------------#
history = model.fit(X_train, y_train, epochs= epocas, batch_size= batch,
    validation_data= (X_val, y_val), callbacks=[ckpt, reduce_lr, early_stop])

df_hist = pd.DataFrame(history.history)

#------------------------------------------------------------------------------------------------#
output_file = os.path.join('resultados/training.csv')
df_hist.to_csv(output_file, index=False)

**RESULTS VISUALIZATION**

In [ ]:
# training loss vs validation loss
#------------------------------------------------------------------------------------------------#
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylim( 0,  max(history.history['val_loss']) )
plt.legend()

plt.savefig(os.path.join("resultados/val_loss.png"), dpi= dpi, bbox_inches='tight')
plt.show()



# training mae vs validation mae in °C
#------------------------------------------------------------------------------------------------#
plt.plot(history.history['mae_c'], label='Training MAE (°C)')
plt.plot(history.history['val_mae_c'], label='Validation MAE (°C)')
plt.xlabel('Epochs')
plt.ylim(0, max(history.history['val_mae_c']))
plt.legend()

plt.savefig(os.path.join("resultados/val_mae_c.png"), dpi=dpi, bbox_inches='tight')
plt.show()

In [ ]:
# TEST PREDICTIONS

y_pred_scaled = model.predict(X_test)  # predict on scaled data
#------------------------------------------------------------------------------------------------#

# reverse scaling:
y_pred_2d = y_pred_scaled.reshape(len(y_pred_scaled), -1)  # reshape to 2d: (n, 1)
y_pred = scaler_y.inverse_transform(y_pred_2d).flatten() # reverse scaling and flatten

# same for y_test:
y_test_2d = y_test.reshape(len(y_test), -1)
y_test  = scaler_y.inverse_transform(y_test_2d).flatten()

#------------------------------------------------------------------------------------------------#
# save predictions
df_preds = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test})
df_preds.to_csv(os.path.join("resultados/predicciones.csv"), index=False)


#------------------------------------------------------------------------------------------------#
print(f"y_pred shape: {y_pred.shape}")
print(f"y_test shape: {y_test.shape}")
print("X_test shape:", X_test.shape)


In [ ]:
# POST-HOC LINEAR CALIBRATION (VAL -> FITS TEST) --- OPTIONAL BUT VERY USEFUL
#------------------------------------------------------------------------------------------------#

# val in °C
y_val_pred_scaled = model.predict(X_val)
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1,1)).flatten()
y_val_true = scaler_y.inverse_transform(y_val.reshape(-1,1)).flatten()

#------------------------------------------------------------------------------------------------#
# linear fit on validation
a, b = np.polyfit(y_val_pred, y_val_true, 1)

# apply to test (overwrites y_pred already in °C)
y_pred = a*y_pred + b
#------------------------------------------------------------------------------------------------#


# re-save predictions (so the metrics block reads them already calibrated)
df_preds = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test})
df_preds.to_csv(os.path.join("resultados/predicciones_cal.csv"), index=False)


In [ ]:
# PREDICTION METRICS ON REAL SCALE (°C) - TEST

df = pd.read_csv('resultados/predicciones.csv')
df_cal = pd.read_csv('resultados/predicciones_cal.csv')

pred = df['y_pred']
pred_cal = df_cal['y_pred']

test = df['y_test']

mse = mean_squared_error(test, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, pred)

mse_cal = mean_squared_error(test, pred_cal)
rmse_cal = np.sqrt(mse_cal)
mae_cal = mean_absolute_error(test, pred_cal)

print(f"RMSE: {np.round(rmse, 5)}")
print(f"MAE: {np.round(mae, 5)}")

print(f"RMSE_cal: {np.round(rmse_cal, 5)}")
print(f"MAE_cal: {np.round(mae_cal, 5)}")


#------------------------------------------------------------------------------------------------#
with open('resultados/metricas_°C.txt', 'w') as f:
    f.write("RMSE\tMAE\n")
    f.write(f"{np.round(rmse, 7)}\t{np.round(mae, 7)}\n")

with open('resultados/metricas_°C_cal.txt', 'w') as f:
    f.write("RMSE\tMAE\n")
    f.write(f"{np.round(rmse, 7)}\t{np.round(mae, 7)}\n")

In [ ]:
# PREDICTION PLOTS

#------------------------------------------------------------------------------------------------#
plt.scatter(y_test, y_pred,  s=2)  # pred vs test
plt.xlabel('y_test (° C)')
plt.ylabel('y_pred (° C)')
plt.title('Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red')  # reference line
plt.savefig(os.path.join('resultados/Predictions_cal.png'), format='png', dpi= dpi, bbox_inches='tight')
plt.show()


#------------------------------------------------------------------------------------------------#
residuos = y_test - y_pred  # residuals

plt.scatter(y_test, residuos, s=2)  # residuals plot
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('y_test  (° C)')
plt.ylabel('Residuals = y_test - y_pred')
plt.ylim(min(residuos), max(residuos))
plt.savefig(os.path.join('resultados/Residuals.png'), format='png', dpi= dpi, bbox_inches='tight')
plt.show()


#------------------------------------------------------------------------------------------------#
residuos_fft = fft(residuos)  # residuals fft
frecuencias = fftfreq(len(residuos))

plt.plot(frecuencias[:len(frecuencias)//2], np.abs(residuos_fft[:len(residuos_fft)//2]))
plt.xlabel('Frequency')
plt.title('Residuals FFT')
plt.show()

In [ ]:
# TEST CYCLE PREDICTIONS (TEMPERATURE CURVE)

temps = pd.read_csv(os.path.join('temperature_values.csv')).iloc[:, 0].values

true_test = temps[num_train_test:]
pred_test = y_pred

n = min(len(true_test), len(pred_test)) # adjust to same length if there is a shift
true_test = true_test[:n]
pred_test = pred_test[:n]

x = range(num_train_test, num_train_test + n) # index (spectrum number)


#--------------------------------------------------------------------------------------------------------------------------------------------------------------
plt.figure(figsize=(10, 4)) # plot
plt.plot(x, true_test, linestyle='-', linewidth=1.5, label='Measured temperature')
plt.scatter(x, pred_test,  color='red', marker='o', s=0.3,   label='Predicted temperature')
plt.xlabel('Spectrum number')
plt.ylabel('Temperature (°C)')
plt.legend()

f_out = pd.DataFrame({'indice': x, 'temp_real':true_test, 'temp_predicha': pred_test }) # save test predictions (°C)
f_out.to_csv(os.path.join('resultados/T_vs_Pred.csv'), index=False)


plt.savefig(os.path.join('resultados/T_vs_Pred.png'), dpi=300)
plt.show()


In [ ]:
# DISTRIBUTION OF y_pred-y_test (HISTOGRAM)

df = pd.read_csv('resultados/predicciones_cal.csv')

df['diff'] = (df['y_pred'] - df['y_test'])

std_diff = df['diff'].std()
print(f'Standard deviation of differences: {std_diff:.4f}')


#---------------------------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(df['diff'], bins=35)
plt.xlabel('y_pred - y_test (°C)')
plt.ylabel('Frequency')

mean_diff = df['diff'].mean()

plt.axvline(mean_diff, color='red', linestyle='--', linewidth=1.5)
plt.axvline(0, color='green', linestyle='--', linewidth=1.5)


x_max, y_max = plt.xlim()[1], plt.ylim()[1]
texto = f"Mean: {mean_diff:.3f} °C\nStd: {std_diff:.3f} °C"
plt.text(x_max-0.2, y_max-10, texto, ha='right', va='top', fontsize=11,
         bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

plt.savefig("resultados/err_dist.png", dpi= dpi)

plt.show()

In [ ]:
#---------------------------------------------------------------------------------------------
# OFFSET CORRECTION ON TEST SET
#---------------------------------------------------------------------------------------------


# offset in °C
offset = df['diff'].mean()
print(f"Offset (mean of y_pred - y_test): {offset:.5f} °C")


#---------------------------------------------------------------------------------------------
# corrected predictions (removing the offset)
df['y_pred_corr'] = df['y_pred'] - offset
df['diff_corr'] = df['y_pred_corr'] - df['y_test']  # new error: y_pred_corr - y_test

mean_diff_corr = df['diff_corr'].mean()
std_diff_corr  = df['diff_corr'].std()
rmse_corr = np.sqrt(mean_squared_error(df['y_test'], df['y_pred_corr'])) # rmse on corrected predictions
mae_corr  = mean_absolute_error(df['y_test'], df['y_pred_corr'])         # mae on corrected predictions


#---------------------------------------------------------------------------------------------
print(f"Mean after offset correction: {mean_diff_corr:.5f} °C")
print(f"Std after offset correction:   {std_diff_corr:.5f} °C")
print(f"Corrected RMSE: {rmse_corr:.5f} °C")
print(f"Corrected MAE:  {mae_corr:.5f} °C")

In [ ]:
# SAVE NEW PREDICTIONS AND METRICS WITH CORRECTED OFFSET

df.to_csv("resultados/predicciones_calibradas_corregidas.csv", index=False)

with open('resultados/metricas_°C_cal_corr.txt', 'w') as f:
    f.write("offset(°C)\tRMSE_corr\tMAE_corr\tmean_diff_corr\tstd_diff_corr\n")
    f.write(f"{offset:.7f}\t{rmse_corr:.7f}\t{mae_corr:.7f}\t"
            f"{mean_diff_corr:.7f}\t{std_diff_corr:.7f}\n")

In [ ]:
# Histogram (y_pred - y_test) after offset correction

plt.figure(figsize=(8, 5))
plt.hist(df['diff_corr'], bins=30)
plt.xlabel('y_pred_corr - y_test (°C)')
plt.ylabel('Frequency')

plt.axvline(0, color='green', linestyle='--', linewidth=2)

x_max, y_max = plt.xlim()[1], plt.ylim()[1]
texto = f"Media: {mean_diff_corr:.3f} °C\nStd: {std_diff_corr:.3f} °C"
plt.text(x_max - 0.2, y_max - 10, texto, ha='right', va='top', fontsize=11,
         bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

plt.savefig("resultados/err_dist_corr.png", dpi= dpi)
plt.show()

In [ ]:
# MEASURED vs PREDICTED T CURVE - OFFSET CORRECTED - TEST CYCLE

n = len(df)  # number of test samples

true_test = df['y_test'].values
pred_test_corr = df['y_pred_corr'].values

x = np.arange(n)


#---------------------------------------------------------------------------------------------
plt.figure(figsize=(10, 4))
plt.plot(x, true_test, linestyle='-', linewidth=1.5, label='Measured temperature')
plt.scatter(x, pred_test_corr, color='red', marker='o', s=0.3, label='Predicted temperature (corr.)')
plt.xlabel('Spectrum index')
plt.ylabel('Temperature (°C)')
plt.legend()

f_out_corr = pd.DataFrame({'indice': x, 'temp_real': true_test, 'temp_pred_corrida': pred_test_corr})
f_out_corr.to_csv(os.path.join('resultados/T_vs_Pred_corr.csv'), index=False)

plt.savefig(os.path.join('resultados/T_vs_Pred_corr.png'), dpi=300)
plt.show()

**SAVING**

In [ ]:
# SAVE MODEL, SCALERS AND METADATA

#------------------------------------------------------------------------------------------------#
model_path = os.path.join(f"resultados/CNN-trained_{epocas}ep_{λ_min}-{λ_max}.h5")
model.save(model_path)

#------------------------------------------------------------------------------------------------#
# save scalers
joblib.dump(scaler_x, "resultados/scaler_x.pkl")  # intensities
joblib.dump(scaler_y, "resultados/scaler_y.pkl")  # temperatures

#------------------------------------------------------------------------------------------------#
# save useful data
meta = {
    "λ_min": float(λ_min),
    "λ_max": float(λ_max),
    "offset_T":   float(offset), # in °C

    "model": {
        "type": "CNN",
        "CNN_L2": float(CNN_L2),
        "Conv_Act": str(Conv_Act),
        "Dense_Act": str(Dense_Act),
        "Filters": [int(x) for x in Filters],
        "Kernels": [int(x) for x in Kernels],
        "Pool": [int(x) for x in Pool],
        "Att_Units": int(Att_Units),
        "Att_Heads": int(Att_Heads),
        "Dense_Units": [int(x) for x in Dense_Units]   } }

    # add any other relevant data


with open("resultados/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)


**DOWNLOAD RESULTS AND CLEAR SESSION**

In [ ]:
# DOWNLOAD RESULTS
!zip -r Results.zip resultados

files.download('Results.zip')

In [ ]:
import shutil
# CLEAR COLAB SESSION

# delete all folders and files (except the notebook)
for nombre in os.listdir():
    if nombre != 'sample_data':  # skip default colab folder
        try:
            ruta = os.path.join(os.getcwd(), nombre)
            if os.path.isfile(ruta):
                os.remove(ruta)
            elif os.path.isdir(ruta):
                shutil.rmtree(ruta)
        except Exception as e:
            print(f"Error deleting {nombre}: {e}")

print("Files and folders deleted from session.")